<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part E: Wrap-Up and Integration</h2>
<h2>Notebook E01: An End-to-End Forecasting Pipeline</h2>
</div>

Every notebook so far isolated one idea. This one puts them in order on a problem none of them has seen,
and the order is most of the point: the decisions that determine whether a forecasting project succeeds
are made before any model is fitted.

The case study is **day-ahead electricity load for Belgium**, hourly, from 2015 to 2020. We train on
everything up to the end of 2019 and forecast through 2020, which turns out to be the most instructive
choice in the notebook.

> Needs the optional deep learning group for section 6: `uv sync --group dl`. The rest runs on the
> default install.

---

**Contents**

1. [The Problem](#1.-The-Problem)
2. [Understand the Data](#2.-Understand-the-Data)
3. [Decide How You Will Be Judged](#3.-Decide-How-You-Will-Be-Judged)
4. [Baselines](#4.-Baselines)
5. [A Feature-Based Model](#5.-A-Feature-Based-Model)
6. [A Sequence Model](#6.-A-Sequence-Model)
7. [When the World Changes](#7.-When-the-World-Changes)
8. [What to Ship, and What to Watch](#8.-What-to-Ship,-and-What-to-Watch)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-The-Problem">1. The Problem</h3>
</div>

Before any data is loaded, four questions have answers, and they constrain everything that follows.

**What is being decided?** A grid operator schedules generation a day ahead. The forecast has to cover the
next 24 hours and it has to exist before the market closes, so anything the model needs must be knowable
by then.

**What is the horizon?** 24 hours, in one shot. Not one hour, which is a far easier problem and a
different model.

**What does an error cost?** Under-forecasting means buying power late and expensively; over-forecasting
means paying for reserve that goes unused. The costs are not symmetric in reality, but they are closer to
symmetric than squared error implies, so mean absolute error is the honest default here.

**What is the alternative?** Somebody is already producing this forecast, and the fallback is "yesterday's
profile". That is the number to beat, and it is the reason Notebook
[A05](./A05_Forecasting_baselines.ipynb) exists.

Write these down before starting. Every later choice, the split, the metric, the feature set, is answerable
to them.

In [ ]:
import importlib.util
import logging
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

import nb_config

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

NEURALFORECAST_AVAILABLE = importlib.util.find_spec("neuralforecast") is not None

if NEURALFORECAST_AVAILABLE:
    import torch
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS

    torch.set_num_threads(1)
    logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
    logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

print(f"Deep learning section available: {NEURALFORECAST_AVAILABLE}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Understand-the-Data">2. Understand the Data</h3>
</div>

Notebooks [A01](./A01_Loading_data.ipynb) to [A04](./A04_Handling_outliers.ipynb) in about fifteen
minutes: load it, look at it, check whether it is broken, and find out what structure it has.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "BE") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .asfreq("h")
)

expected = pd.date_range(load.index.min(), load.index.max(), freq="h")

print(f"{len(load):,} hourly observations, "
      f"{load.index.min().date()} to {load.index.max().date()}")
print(f"Missing timestamps: {len(expected.difference(load.index))}")
print(f"Missing values:     {int(load.isna().sum())}")
print(f"Range: {load.min():.0f} to {load.max():.0f} MW")

No gaps and no missing values, which is worth two minutes of checking and is not always the answer. The
imputation machinery of Notebook [A03](./A03_Handling_missing_data.ipynb) is not needed here, and knowing
that quickly is the point of checking.

Outliers next. The right question is not "which points are extreme" but "which points are extreme *given
what we expect for that hour of that weekday*", which is the residual approach from Notebook
[A04](./A04_Handling_outliers.ipynb).

In [ ]:
profile = load.groupby([load.index.dayofweek, load.index.hour]).transform("median")
residual = load - profile

q1, q3 = residual.quantile([0.25, 0.75])
iqr = q3 - q1
extreme = (residual < q1 - 3 * iqr) | (residual > q3 + 3 * iqr)

print(f"Points beyond 3x IQR of the weekly profile: {int(extreme.sum())}")
print()
print("Largest shortfalls against the weekly profile:")
print(residual.nsmallest(6).round(0).to_string())

Nothing is flagged by the rule, and the largest deviations are dated rather than random: **Easter Monday**
and **1 May**. They are public holidays, when a country's electricity demand looks like a Sunday.

That is the A04 lesson arriving in a real project. These are not errors to clean, they are structure to
model, and the correct response is a feature rather than an imputation. We will add one in section 5.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 7))

recent = load["2019-01-01":"2019-01-21"]
axes[0, 0].plot(recent.index, recent.values, color="steelblue", linewidth=1.0)
axes[0, 0].set_title("Three weeks", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Load (MW)")
axes[0, 0].tick_params(axis="x", rotation=30)

by_hour = load.groupby(load.index.hour).mean()
axes[0, 1].plot(by_hour.index, by_hour.values, color="seagreen", marker="o", markersize=4)
axes[0, 1].set_title("Average by hour of day", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Hour")

by_weekday = load.groupby(load.index.dayofweek).mean()
axes[1, 0].plot(by_weekday.index, by_weekday.values, color="darkorange", marker="o", markersize=5)
axes[1, 0].set_title("Average by day of week (0 = Monday)", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Day")
axes[1, 0].set_ylabel("Load (MW)")

yearly = load.resample("MS").mean()
for year in range(2015, 2021):
    series = yearly[yearly.index.year == year]
    axes[1, 1].plot(series.index.month, series.values, marker="o", markersize=3, label=str(year))
axes[1, 1].set_title("Monthly average by year", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Month")
axes[1, 1].legend(fontsize=8, ncol=2)

for ax in axes.ravel():
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Three seasonalities, exactly as Notebook [B03](./B03_Advanced_statistical_models.ipynb) described: a daily
shape, a weekday/weekend split, and a yearly cycle with winter demand well above summer.

The bottom-right panel contains something else. **2020 sits below every other year**, and the gap is
widest in spring. Hold that thought; section 7 is about it.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Decide-How-You-Will-Be-Judged">3. Decide How You Will Be Judged</h3>
</div>

This is the step people skip, and skipping it is how projects end up with a number nobody can defend.
Fix the protocol **now**, before seeing any model's score, so that no choice is made to flatter a result.

**Split by time, never at random.** Train on everything up to the end of 2019; forecast every day of 2020.

**Match the evaluation to the use.** The model will be asked, each morning, for the next 24 hours. So we
evaluate exactly that: one forecast per day, 24 hours ahead, with history refreshed but the model *not*
refitted. This is the rolling-origin evaluation of Notebook [A06](./A06_Evaluating_models.ipynb).

**One metric, chosen in advance.** MAE, in megawatts, for the reasons in section 1.

**Two baselines, not one.** Yesterday's profile, and the same weekday last week. The second matters because
the first fails predictably every Monday and every holiday.

In [ ]:
TRAIN_END = "2019-12-31"
HORIZON = 24

train = load[:TRAIN_END]
test = load["2020-01-01":]

print(f"Train: {train.index.min().date()} to {train.index.max().date()}  ({len(train):,} hours)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()}  ({len(test):,} hours)")


def evaluate_daily(forecast_function, name, verbose=True):
    """One forecast per day of the test period, scored on the next 24 hours."""
    started = time.time()
    records = []

    last_full_day = load.index.max().normalize() - pd.Timedelta(days=1)
    for origin in pd.date_range("2020-01-01", last_full_day, freq="D"):
        actual = load.loc[origin:origin + pd.Timedelta(hours=HORIZON - 1)]
        if len(actual) < HORIZON:
            continue

        predicted = forecast_function(origin)
        if predicted is None:
            continue

        records.append({
            "date": origin,
            "mae": mean_absolute_error(actual.values, np.asarray(predicted)[:HORIZON]),
        })

    daily = pd.DataFrame(records).set_index("date")
    if verbose:
        print(f"{name:<22} MAE {daily['mae'].mean():7.1f} MW   "
              f"({len(daily)} days, {time.time() - started:.0f}s)")
    return daily["mae"]

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Baselines">4. Baselines</h3>
</div>

Baselines first, always. They cost nothing, they cannot be overfitted, and until one exists no model
score means anything.

In [ ]:
def repeat_offset(days):
    """Forecast by copying the same hours a fixed number of days earlier."""
    def forecast(origin):
        start = origin - pd.Timedelta(days=days)
        window = load.loc[start:start + pd.Timedelta(hours=HORIZON - 1)]
        return window.values if len(window) == HORIZON else None
    return forecast


results = {}
results["Naive: yesterday"] = evaluate_daily(repeat_offset(1), "Naive: yesterday")
results["Naive: last week"] = evaluate_daily(repeat_offset(7), "Naive: last week")

Two baselines, and they disagree about which is better by a wide margin: repeating last week beats
repeating yesterday by more than a hundred megawatts.

That is the weekly cycle asserting itself. Yesterday's profile is wrong every Monday, wrong every Saturday,
and wrong on every holiday, and those errors are large enough to dominate the average. The same-weekday
baseline gets the shape right and only misses the level.

Against a mean load of about 9,800 MW, the better baseline is off by around 4%. Any model we build now has
a number to beat, and it is not a soft one.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-A-Feature-Based-Model">5. A Feature-Based Model</h3>
</div>

Part C in one cell. Lags at the offsets the structure suggests, a rolling mean for the recent level,
calendar terms for the three seasonalities, and a **holiday flag**, which section 2 told us we would need.

Every lag is at least 24 hours old, because a forecast made this morning for tomorrow cannot use anything
from today. That constraint is what Notebook [C01](./C01_Feature_engineering.ipynb) spent a section on,
and here it is the difference between a model that works and one that cannot be deployed.

In [ ]:
BELGIAN_HOLIDAYS = pd.to_datetime([
    # A few years of fixed and moving public holidays, enough for this dataset
    "2015-01-01", "2015-04-06", "2015-05-01", "2015-05-14", "2015-05-25", "2015-07-21",
    "2015-08-15", "2015-11-01", "2015-11-11", "2015-12-25",
    "2016-01-01", "2016-03-28", "2016-05-01", "2016-05-05", "2016-05-16", "2016-07-21",
    "2016-08-15", "2016-11-01", "2016-11-11", "2016-12-25",
    "2017-01-01", "2017-04-17", "2017-05-01", "2017-05-25", "2017-06-05", "2017-07-21",
    "2017-08-15", "2017-11-01", "2017-11-11", "2017-12-25",
    "2018-01-01", "2018-04-02", "2018-05-01", "2018-05-10", "2018-05-21", "2018-07-21",
    "2018-08-15", "2018-11-01", "2018-11-11", "2018-12-25",
    "2019-01-01", "2019-04-22", "2019-05-01", "2019-05-30", "2019-06-10", "2019-07-21",
    "2019-08-15", "2019-11-01", "2019-11-11", "2019-12-25",
    "2020-01-01", "2020-04-13", "2020-05-01", "2020-05-21", "2020-06-01", "2020-07-21",
    "2020-08-15", "2020-11-01", "2020-11-11", "2020-12-25",
])


def build_features(series):
    """Everything knowable the day before, and nothing else."""
    features = pd.DataFrame(index=series.index)

    # The shortest lag is 24 hours: a forecast for tomorrow cannot see today
    for lag in (24, 25, 48, 168, 169):
        features[f"lag_{lag}"] = series.shift(lag)

    features["roll_mean_168"] = series.shift(24).rolling(168).mean()

    features["hour"] = series.index.hour
    features["day_of_week"] = series.index.dayofweek
    features["month"] = series.index.month
    features["day_of_year"] = series.index.dayofyear
    features["is_weekend"] = (series.index.dayofweek >= 5).astype(int)
    features["is_holiday"] = series.index.normalize().isin(BELGIAN_HOLIDAYS).astype(int)

    return features


features = build_features(load)
usable = features.notna().all(axis=1) & load.notna()

X_train = features[usable & (features.index <= TRAIN_END)]
y_train = load[usable & (load.index <= TRAIN_END)]

started = time.time()
gradient_boosting = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.05, num_leaves=63, random_state=0, verbose=-1
).fit(X_train, y_train)

print(f"{X_train.shape[1]} features, {len(X_train):,} training rows, "
      f"trained in {time.time() - started:.0f}s")


def gradient_boosting_forecast(origin):
    wanted = pd.date_range(origin, periods=HORIZON, freq="h")
    window = features.reindex(wanted)
    return None if window.isna().any().any() else gradient_boosting.predict(window)


results["LightGBM"] = evaluate_daily(gradient_boosting_forecast, "LightGBM")

285.5 MW, against the better baseline's 412.4: nearly a third of the error removed, from a model that
trains in three seconds on features we could describe in a sentence.

This is the Part C result repeating on a new problem. The gains came from lags at the right offsets, the
calendar, and one holiday flag, which is to say from **knowing what the series is**, not from the
algorithm. LightGBM with default-ish settings was enough.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-A-Sequence-Model">6. A Sequence Model</h3>
</div>

Part D in one cell, using the architecture that won it. N-BEATS reads the raw sequence, so it gets no
holiday flag and no calendar features at all: a fair test of whether learned structure beats engineered
structure on the same problem.

> **This section takes a few minutes and needs `uv sync --group dl`.** The notebook continues without it.

In [ ]:
if NEURALFORECAST_AVAILABLE:
    frame = pd.DataFrame({
        "unique_id": "BE",
        "ds": load.index,
        "y": load.values.astype(np.float32),
    }).dropna()

    started = time.time()
    neural = NeuralForecast(
        models=[NBEATS(h=HORIZON, input_size=168, max_steps=300,
                       enable_progress_bar=False, random_seed=0)],
        freq="h",
    )
    neural.fit(frame[frame["ds"] <= TRAIN_END], val_size=24 * 90)
    print(f"N-BEATS trained in {time.time() - started:.0f}s")

    def neural_forecast(origin):
        history = frame[frame["ds"] < origin]
        if len(history) < 200:
            return None
        return neural.predict(df=history)["NBEATS"].values

    results["N-BEATS"] = evaluate_daily(neural_forecast, "N-BEATS")

248.2 MW, about 13% better than the feature-based model, from a network that was given no calendar, no
holiday flag and no lags: only the raw sequence of the previous week.

It inferred the daily and weekly shape from examples, which is the trade Part D described. Whether that
trade is worth making is not obvious from this number alone, and section 7 is where it becomes clear.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-When-the-World-Changes">7. When the World Changes</h3>
</div>

Now the part that makes this a case study rather than an exercise.

Every model here was trained on data ending 31 December 2019 and asked to forecast 2020. In March of that
year, Belgian electricity demand fell by about a tenth and stayed there for months, for reasons no amount
of history could have anticipated.

This is a **structural break**, and it is the most common way a working forecasting system fails. Not
through a bad choice of architecture, but because the process generating the data stopped being the
process the model learned.

In [ ]:
scores = pd.DataFrame(results)
monthly = scores.resample("MS").mean()

monthly.round(0)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for column in scores.columns:
    axes[0].plot(monthly.index, monthly[column], marker="o", markersize=5,
                 linewidth=1.5, label=column)
axes[0].axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-05-31"),
                color="crimson", alpha=0.10)
axes[0].text(pd.Timestamp("2020-03-18"), axes[0].get_ylim()[1] * 0.95,
             "lockdown", fontsize=9, color="crimson", va="top")
axes[0].set_title("Forecast error by month, 2020", fontsize=13, fontweight="bold")
axes[0].set_ylabel("MAE (MW)")
axes[0].legend(fontsize=9)

actual_monthly = load["2020"].resample("MS").mean()
reference = load[:"2019"].groupby(load[:"2019"].index.month).mean()
axes[1].plot(actual_monthly.index, actual_monthly.values, color="black",
             marker="o", markersize=5, linewidth=1.8, label="2020 actual")
axes[1].plot(actual_monthly.index, reference.loc[actual_monthly.index.month].values,
             color="gray", linestyle="--", linewidth=1.5, label="2015-2019 average")
axes[1].axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-05-31"),
                color="crimson", alpha=0.10)
axes[1].set_title("What the models were up against", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Mean load (MW)")
axes[1].set_xlabel("Month of 2020")
axes[1].legend(fontsize=9)

for ax in axes:
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The monthly table is the most useful thing in this notebook, and it says three things that the overall
averages hide completely.

**Before the break, the two models are level.** In January LightGBM scores 271 against N-BEATS's 335; in
February the order reverses, 277 against 228. On two months of evidence you would ship the simpler,
faster, more interpretable one, and you would be making a defensible choice.

**Through the break, they come apart completely.** March pushes LightGBM to 351, and April to **462**,
while N-BEATS holds at 267 and 275. The feature-based model ends up 70% worse than it was in January, and
it is beaten in April by *both* baselines: the same-weekday forecast scores 378 and yesterday's profile
406. A model that had been removing a third of the baseline's error spent the worst month of the year
adding to it.

**And it recovers.** By May LightGBM is back to 263, and from June onward the two models are level again.
The damage is confined almost exactly to the months when the level was moving, which is the signature of a
break rather than of a broken model.

The reason is visible in what each one uses. LightGBM was given `month` and `day_of_year`, and learned from
five years of history what April looks like. April 2020 did not look like that, and the model had no way to
know. N-BEATS reads the previous 168 hours and normalises them, so a level that dropped a tenth in March
arrived as its input rather than as a contradiction of its training data. **A model that keys off the
recent window adapts to a level shift; a model that keys off the calendar cannot.**

**And the baselines got *better*.** Yesterday's profile scores 563 in January and 406 in April. Lockdown
demand was flatter and more repetitive, which made the naive forecast's job easier at the same moment it
made the models' job harder. Both effects push in the same direction, and together they are what let a
baseline overtake a trained model.

That is the point to carry out of this course. Over the year, LightGBM looks like a solid second place at
285 against N-BEATS's 248. Within the year, there is a month where it was the worst forecaster in the
table, including the two that take one line to write. **The annual average contains no hint of it.**

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-What-to-Ship,-and-What-to-Watch">8. What to Ship, and What to Watch</h3>
</div>

**What to ship.** On this evidence, N-BEATS: it wins overall, it holds up when conditions change, and a
minute of training is not a burden. LightGBM at 285 MW would be defensible if interpretability or a
light dependency footprint mattered more, and outside the break the two are level — but you would be
choosing a model that has demonstrated how it fails, which is worth knowing before rather than after.

Notice that this is a decision, with reasons, rather than a ranking. The reasons are the useful part.

**What to watch.** A deployed forecaster needs three things monitored, and this notebook has demonstrated
why each one:

| Watch | Because |
|---|---|
| **Error against the live baseline** | The model's edge can vanish without its error rising |
| **Error by segment**, month, weekday, holiday | Averages hid a 27% degradation for a whole month |
| **The input distribution**, not just the error | The level shifted in March; the error followed |

**Retrain on a schedule, and on a trigger.** A schedule handles drift. A trigger, error exceeding its
recent range for several days, handles breaks. Neither alone is enough.

**The checklist this course has been building:**

1. Write down the decision, the horizon, and the cost of an error, before touching the data.
2. Look at the series. Check for gaps and for values that are structure rather than error.
3. Fix the evaluation protocol before fitting anything, and make it match how the forecast will be used.
4. Build a baseline. Two, if the series has more than one obvious rhythm.
5. Fit the simplest model that could work, and only then something more expensive.
6. Compare on held-out data, at several origins, on a metric chosen in advance.
7. Look at the error broken down, not just averaged.
8. Decide what to ship, say why, and say what would change your mind.

The models in step 5 will be different in five years. Steps 1 to 4 and 6 to 8 will not.

---

That is the course. You can take a series you have never seen, work out whether it is forecastable, clean
it without damaging it, build a baseline that is hard to beat, choose a model family for reasons you can
state, evaluate it in a way that will not mislead you, and say what you would watch after it is deployed.

The models will keep changing. The order of those steps will not.